In [ ]:
import HybridDynamics as HD
import Plots as plt
using LaTeXStrings

This notebook runs Fuller's problem as a demonstration of a Filippov system. This is the discontinuous system on the plane given by
$$
\dot{x}_1 = x_2, \quad \dot{x}_2 = u,
$$
where
$$ u = \begin{cases}
+1, & x_1 < -\mathrm{sign}(x_2)\xi x_2^2 \\
-1, & x_1 > -\mathrm{sign}(x_2)\xi x_2^2
\end{cases}$$
It is known that trajectories are chattering and approach the origin.

In [ ]:
# defining the Fuller constant
const ξ = sqrt((sqrt(33) - 1) / 24)

In [ ]:
# Define the vector fields
# F is active when H(x) > 0 (Region G_ -> u = -1)
function F(x)
    return [x[2], -1.0]
end

In [ ]:
# G is active when H(x) < 0 (Region G_+ -> u = +1)
function G(x)
    return [x[2], 1.0]
end

In [ ]:
# Defining the guard
function H(x)
    return x[1] + sign(x[2]) * ξ * x[2]^2
end

In [ ]:
# Define the Normal
function N(x)
    return [1.0, 2.0 * ξ * abs(x[2])]
end

In [ ]:
sys = HD.FilippovSystem(F, G, H, N)

In [ ]:
# Setting up two trajectories to plot later!
x0_1 = [0.15, 0.35]
prob1 = HD.prob(sys, x0_1, (0.0, 5.0))
sol1 = HD.solve(prob1, HD.RK4())

x0_2 = [-0.15, -0.35] 
prob2 = HD.prob(sys, x0_2, (0.0, 5.0))
sol2 = HD.solve(prob2, HD.RK4())


In [ ]:
# Extracting states for both trajectories. 
x1 = [state[1] for state in sol1.x]
y1 = [state[2] for state in sol1.x]

x2 = [state[1] for state in sol2.x]
y2 = [state[2] for state in sol2.x];

In [ ]:
# Generating the switchig manifold (guard) over what the plot will show
y_surf = range(-0.5, 0.5, length=200)
x_surf = [-sign(y) * ξ * y^2 for y in y_surf];


In [ ]:
# Making the plot. 
plt.plot(x1, y1, 
    title="Fuller's Problem Phase Portrait", 
    label="Trajectory 1", 
    lw=1.5, color=:green,
    framestyle=:origin,      # Forces the x and y axes to cross at (0,0)
    xlims=(-0.15, 0.15),     # Narrows the X bounds to stretch the curves horizontally (so we can see what happens)
    ylims=(-0.5, 0.5),       # Keeps the Y bounds tall
    grid=true,
    legend=:topright
)
plt.plot!(x2, y2, label="Trajectory 2", lw=1.5, color=:blue)
plt.plot!(x_surf, y_surf, label="Switching Manifold H(x)=0", color=:black, lw=1.5)